I think everything up to some point idk. I have to do this cause i really want to test the GP code on some real tickers. 

GP is only being applied to the high return portfolio. I don't think it is necessary for the low volatility portfolio to be completely honest. 

### IMPORTANT 

It is becoming clear to me that hyperoptimizing our parameters does not have a significant effect on our winrate in validation periods. Why? Because our ticker list is like 32 ish tickers. When the GP runs with the upper bound of 20, chances are the parameters that allow for 20 tickers out of the 32 to pass the indicator filter are not that hyperoptimized for returns but rather just so we can get 20 tickers pass. It is purely for this reason that I STRONGLY believe the way we weigh each ticker likely matters much more for maximizing returns than the tickers that we let through.


Play around with this, try not to break it. Test indicators. But more importantly, we need an actual weighing strategy. That along with edge case proofing the code should be our top priorities.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from skopt import gp_minimize
from skopt.space import Real

In [135]:
#Constants for dates

start_dl = "2023-01-01"
end_dl = "2026-11-20"

### PIPELINE PART A: DATA DOWNLOAD

In [136]:

# INITIAL PORTFOLIO FILTER
# 1. Must be a company stock listed in the US or Canada
# 2. Must trade in USD or CAD
# 3. Average volume must be at least 5000
# 4. Exclude months with fewer than 18 trading days

# Read the tickers
try:
    tickers = pd.read_csv("Tickers_Example.csv", header=None)
    tickers = tickers.dropna()
except Exception as e:
    raise ValueError("Could not read ticker file") from e

eligible = []
eligible_currency = []
checked = []  # NEW: track tickers already checked
stock_info = {}  # NEW: save stock information for later

for i in range(len(tickers)):
    # NEW: remove spaces and make tickers uppercase
    symbol = str(tickers.iloc[i, 0]).strip().upper()

    # NEW: skip empty or duplicate tickers
    if symbol == "" or symbol in checked:
        continue

    checked.append(symbol)
    valid = True

    try:
        stock = yf.Ticker(symbol)
        info = stock.info

        # Company stock?
        if info.get("quoteType") != "EQUITY":
            valid = False

        # Listed in US/Canada?
        if info.get("market") not in ["us_market", "ca_market"]:
            valid = False

        # Currency in USD/CAD?
        if info.get("currency") not in ["USD", "CAD"]:
            valid = False

        # Check average trading volume
        if valid:
            hist = stock.history(
                start=start_dl,
                end=end_dl
            )

            # NEW: check if volume data is missing
            if hist.empty or "Volume" not in hist.columns:
                valid = False
            else:
                dates = pd.to_datetime(hist.index)
                volume = hist["Volume"].replace(
                    [np.inf, -np.inf], np.nan
                )

                # NEW: remove invalid volume values
                volume = volume.where(volume >= 0)

                days_per_month = volume.groupby(
                    [dates.year, dates.month]
                ).count()

                volume_per_month = volume.groupby(
                    [dates.year, dates.month]
                ).sum()

                total_days = 0
                total_volume = 0

                for m in range(len(days_per_month)):
                    if days_per_month.iloc[m] >= 18:
                        total_days += days_per_month.iloc[m]
                        total_volume += volume_per_month.iloc[m]

                if total_days == 0:
                    valid = False
                elif total_volume / total_days < 5000:
                    valid = False

    # NEW: handle failed ticker requests
    except Exception as e:
        print(symbol, "could not be checked:", e)
        valid = False

    if valid:
        eligible.append(symbol)
        eligible_currency.append(info.get("currency"))
        stock_info[symbol] = info


# Download historical prices
stock_data = {}


if len(eligible) > 0:
    try:
        all_stock = yf.download(
            eligible,
            start=start_dl,
            end=end_dl,
            group_by="ticker",
            auto_adjust=False,
            progress=False
        )

        for ticker in eligible:
            try:
                # NEW: handle single and multiple stock downloads
                if isinstance(all_stock.columns, pd.MultiIndex):
                    if ticker in all_stock.columns.get_level_values(0):
                        indiv_stock = all_stock[ticker].copy()
                    else:
                        indiv_stock = all_stock.xs(
                            ticker, axis=1, level=1
                        ).copy()
                else:
                    indiv_stock = all_stock.copy()

                required = ["High", "Low", "Close"]

                # NEW: check if any price columns are missing
                if not all(col in indiv_stock.columns for col in required):
                    print(ticker, "has missing price columns")
                    continue

                # NEW: remove rows with missing or invalid prices
                indiv_stock = indiv_stock.dropna(subset=required)

                for column in required:
                    indiv_stock = indiv_stock[
                        np.isfinite(indiv_stock[column]) &
                        (indiv_stock[column] > 0)
                    ]

                # NEW: check if enough historical data is available
                if len(indiv_stock) < 252:
                    print(ticker, "does not have enough price data")
                    continue

                # NEW: check if the stock has recent prices
                last_date = pd.Timestamp(indiv_stock.index[-1])
                last_date = last_date.tz_localize(None)

                check_date = min(
                    pd.Timestamp.today().normalize(),
                    pd.Timestamp(end_dl)
                )

                if (check_date - last_date).days > 7:
                    print(ticker, "has outdated price data")
                    continue

                indiv_stock.index = indiv_stock.index.strftime("%Y-%m-%d")
                stock_data[ticker] = indiv_stock

            # NEW: skip stocks with failed downloads
            except (KeyError, TypeError, ValueError) as e:
                print(ticker, "could not be downloaded:", e)

    # NEW: handle failed batch downloads
    except Exception as e:
        print("Historical download failed:", e)

# Convert USD prices to CAD
cadusd = pd.Series(dtype=float)

if len(stock_data) > 0:
    try:
        cadusd = yf.Ticker("CADUSD=X").history(
            start=start_dl,
            end=end_dl
        )["Close"]

        cadusd.index = cadusd.index.strftime("%Y-%m-%d")

        # NEW: remove invalid exchange rates
        cadusd = cadusd.replace([np.inf, -np.inf, 0], np.nan)
        cadusd = cadusd.dropna()

    except Exception as e:
        print("Exchange rates could not be downloaded:", e)


# Convert the remaining stocks
for ticker in list(stock_data):
    table = stock_data[ticker]
    i = eligible.index(ticker)

    if eligible_currency[i] == "USD":
        # NEW: match exchange rates to stock dates
        dates = cadusd.index.union(table.index)
        rates = cadusd.reindex(dates).sort_index().ffill()
        table["CADUSD"] = rates.reindex(table.index)

        table["Close_CAD"] = table["Close"] / table["CADUSD"]
        table["High_CAD"] = table["High"] / table["CADUSD"]
        table["Low_CAD"] = table["Low"] / table["CADUSD"]

    else:
        table["Close_CAD"] = table["Close"]
        table["High_CAD"] = table["High"]
        table["Low_CAD"] = table["Low"]

    required = ["Close_CAD", "High_CAD", "Low_CAD"]

    # NEW: remove invalid CAD prices
    table[required] = table[required].replace(
        [np.inf, -np.inf], np.nan
    )

    table = table.dropna(subset=required)

    for column in required:
        table = table[table[column] > 0]

    # NEW: check that enough valid CAD prices remain
    if len(table) < 252:
        print(ticker, "does not have enough valid CAD prices")
        del stock_data[ticker]
        continue

    # NEW: check for recent CAD prices
    last_date = pd.Timestamp(table.index[-1])

    if (check_date - last_date).days > 7:
        print(ticker, "has outdated CAD prices")
        del stock_data[ticker]
        continue

    stock_data[ticker] = table


# NEW: update eligible lists after removing stocks
final_eligible = []
final_currency = []

for i in range(len(eligible)):
    if eligible[i] in stock_data:
        final_eligible.append(eligible[i])
        final_currency.append(eligible_currency[i])

eligible = final_eligible
eligible_currency = final_currency

print("Eligible tickers:", eligible)
print("Number of eligible stocks:", len(eligible))

Eligible tickers: ['AAPL', 'ABBV', 'ABT', 'ACN', 'AIG', 'AMZN', 'AXP', 'BA', 'BAC', 'BB.TO', 'BIIB', 'BLK', 'BMY', 'C', 'CAT', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PEP', 'PFE', 'RY.TO', 'PM', 'PYPL', 'QCOM', 'SHOP.TO', 'T.TO', 'TD.TO', 'TXN', 'UNH', 'UNP', 'UPS', 'USB']
Number of eligible stocks: 35


### PIPELINE PART B: INDICATOR FUNCTIONS + INDICATOR CALLS

In [137]:
def pull_market_cap(ticker, ticker_df, target_date):

    close_price = ticker_df.loc[target_date:].iloc[0]["Close_CAD"]

    target_dt = pd.to_datetime(target_date)
    end_dt = target_dt + pd.Timedelta(days=5)
    shares_history = ticker.get_shares_full(start=target_dt.strftime("%Y-%m-%d"),
                                            end=end_dt.strftime("%Y-%m-%d"))

    if shares_history is not None and len(shares_history) > 0:
        share_count = shares_history.iloc[0]
    else:
        share_count = ticker.info.get("sharesOutstanding")

    if share_count is None:
        return np.nan
    return close_price * share_count

In [138]:
def calculate_adx(data, period=14):

    if len(data) < 2 * period:
        return pd.DataFrame(np.nan, index=data.index, columns=["adx", "DX", "+DI", "-DI", "smoothed TR"])

    data = data[["High_CAD", "Low_CAD", "Close_CAD"]].copy()

    # 1. True Range
    high_low = data["High_CAD"] - data["Low_CAD"]
    high_close = abs(data["High_CAD"] - data["Close_CAD"].shift(1))
    low_close = abs(data["Low_CAD"] - data["Close_CAD"].shift(1))

    data["Trading Range"] = high_low
    data["up gap"] = high_close
    data["down gap"] = low_close

    data["True Range"] = pd.concat(
        [high_low, high_close, low_close], axis=1
    ).max(axis=1)

    # 2. Directional Momentum
    up_move = data["High_CAD"].diff()
    down_move = -data["Low_CAD"].diff()

    pos_cond = (up_move > down_move) & (up_move > 0)
    neg_cond = (down_move > up_move) & (down_move > 0)

    data["+DM"] = np.where(pos_cond, up_move, 0)
    data["-DM"] = np.where(neg_cond, down_move, 0)

    # 3. Wilder Smoothing
    def wilder_smoothing(series, smoothing_period):
        smoothed = {}
        initial = sum(series.values[1:smoothing_period + 1])
        smoothed[smoothing_period] = initial

        for i in range(smoothing_period + 1, len(series)):
            smoothed[i] = (
                smoothed[i - 1]
                - (smoothed[i - 1] / smoothing_period)
                + series.values[i])

        return pd.Series(list(smoothed.values()), index=series.index[smoothing_period:])

    data["smoothed +DM"] = wilder_smoothing(data["+DM"], period)
    data["smoothed -DM"] = wilder_smoothing(data["-DM"], period)
    data["smoothed TR"] = wilder_smoothing(data["True Range"], period)

    # 4. Directional Index
    data["+DI"] = (data["smoothed +DM"] / data["smoothed TR"]) * 100
    data["-DI"] = (data["smoothed -DM"] / data["smoothed TR"]) * 100

    data["DX"] = (100 * abs(data["+DI"] - data["-DI"])) / (abs(data["+DI"]) + abs(data["-DI"]))

    # 5. Average Directional Index
    adx_values = {}
    initial = data["DX"].iloc[period:2 * period].mean()

    adx_values[2 * period - 1] = initial

    for i in range(2 * period, len(data)):
        adx_values[i] = ((period - 1) * adx_values[i - 1] + data["DX"].iloc[i]) / period

    data["adx"] = pd.Series(list(adx_values.values()), index=data.index[2 * period - 1:])

    data = data[["adx", "DX", "+DI", "-DI", "smoothed TR"]]

    return data

In [139]:
def rolling_price_std(data, period):

    rolling_std = data["Close_CAD"].pct_change(fill_method=None).rolling(period).std()

    return rolling_std

In [140]:
def pull_beta(ticker):

    info = ticker.info
    beta = info.get("beta")

    return beta

In [141]:
ADX_PERIODS = {"1M": 21, "3M": 63, "6M": 126}

def indicator_application(ticker, ticker_df, start_date):
    ticker = yf.Ticker(ticker)

    ticker_df["Market Cap"] = pull_market_cap(ticker, ticker_df,start_date)

    ticker_df["Rolling STD 63d"] = rolling_price_std(ticker_df, 63)

    ticker_df["Beta"] = pull_beta(ticker)

    for label, period in ADX_PERIODS.items():
        adx_df = calculate_adx(ticker_df, period=period)
        adx_df = adx_df.add_suffix(f" {label}")
        
        adx_df[f"DI spread {label}"] = (adx_df[f"+DI {label}"] - adx_df[f"-DI {label}"])
        ticker_df = ticker_df.drop(columns=adx_df.columns, errors="ignore")
        ticker_df = pd.concat([ticker_df, adx_df], axis=1)

    return ticker_df


In [142]:
for ticker, ticker_data in stock_data.items():
    stock_data[ticker] = indicator_application(ticker, ticker_data, end_dl)

IndexError: single positional indexer is out-of-bounds

### PART C: GAUSSIAN PROCESS HYPERPARAMETER OPTIMIZATION

In [ ]:
# basically the length of our competition 
HOLD_DAYS = 6 

series_by_ticker = {}
for ticker, ticker_data in stock_data.items():
  series_by_ticker[ticker] = ticker_data["Close_CAD"]

close = pd.DataFrame(series_by_ticker)

returns = close.pct_change(fill_method=None)

fwd_returns = close.shift(-HOLD_DAYS) / close - 1

print(f"Close Table Shape: {close.shape}")

Condition rule setting.

For High Return Only!!!

In [ ]:
HIGH_RET_RULES = {
    #"adx 6M": ">",
    #"DI spread 6M": ">",
    "adx 3M": ">",
    "DI spread 3M": ">",
    "adx 1M": ">",
    "DI spread 1M": ">",
    "Beta": ">",
    "Market Cap": "<",
}

In [ ]:
def build_tables(rules):

    #essentially evert indicator in rules gets its own column 

    tables = {} 
    for col in rules: 
        col_data = {}
        for t, df in stock_data.items():
            col_data[t] = df[col]
        tables[col] = pd.DataFrame(col_data)
    return tables

high_return_tables = build_tables(HIGH_RET_RULES)
        

Instructions here:

For first_table, always put in the indicator that requires the MOST amount of days
eg. adx_6M > adx_3M

In [159]:
first_table = high_return_tables["adx 3M"]

has_indicators = first_table.notna().any(axis=1)
has_future_return = fwd_returns.notna().any(axis=1)

usable = first_table.index[has_indicators & has_future_return]

test_dates = usable[::-HOLD_DAYS][:100][::-1]

split = int(len(test_dates) * 0.7)
tune_dates = test_dates[:split]
validation_dates = test_dates[split:]

print(len(tune_dates), "tuning weeks:    ", tune_dates[0], "->", tune_dates[-1])
print(len(validation_dates), "validation weeks:", validation_dates[0], "->", validation_dates[-1])

70 tuning weeks:     2024-07-19 -> 2026-03-04
30 validation weeks: 2026-03-12 -> 2026-11-13


Which indicators pass

testing range for Gaussian Process

In [ ]:
def passes(tables, rules, cutoffs, date):
    mask = pd.Series(True, index=close.columns)

    for (col, direction), cutoffs in zip(rules.items(), cutoffs):
        values = tables[col].loc[date]

        if direction == ">":
            this_rule = values > cutoffs 
        else:
            this_rule = values < cutoffs 
            
        mask = mask & this_rule

    return mask 


def make_space(tables, rules, dates):
    space = []
    for col, direction in rules.items():

        if direction == ">":
            q = [0.05, 0.90] 
        else:
            q = [0.10, 0.95]

        lo, hi = tables[col].loc[dates].stack().quantile(q)

        space.append(Real(float(lo), float(hi), name=col))

    return space


hr_space = make_space(high_return_tables, HIGH_RET_RULES, tune_dates)

scoring the tests

In [ ]:

MIN_TICKERS = 10   #setting the punishment for min tickers above the min of 10. I'd rather have more than less. 
MAX_TICKERS = 20   # max 


def weekly_high_return(cutoffs, dates):
    """For each date: pick tickers, hold 6 days, record the return. NaN = too few passed."""
    out = []

    for i in dates:
        passed = passes(high_return_tables, HIGH_RET_RULES, cutoffs, i)

        if passed.sum() < MIN_TICKERS:     # not enough tickers passed this week
            out.append(np.nan)      # mark the week as failed 
            continue 


        # PLACEHOLDER PLACEHOLDER PLACEHOLDER
        # Placeholder logic:
        # If more than 20 tickers pass, take the largest 20 at equal weighting.
        picks = high_return_tables["adx 3M"].loc[i, passed].nlargest(MAX_TICKERS).index
        out.append(fwd_returns.loc[i, picks].mean())

    return pd.Series(out, index=dates)


def count_passing(tables, rules, cutoffs, dates):
    """How many tickers passed the rules on each date."""
    counts = []
    for i in dates:
        counts.append(passes(tables, rules, cutoffs, i).sum())
    return pd.Series(counts, index=dates)


def score_high_return(cutoffs, dates):
    """ONE number for these cutoffs: average weekly return, with penalties for failed weeks."""
    weekly = weekly_high_return(cutoffs, dates)
    n = count_passing(high_return_tables, HIGH_RET_RULES, cutoffs, dates)

    shortfall = (MIN_TICKERS - n).clip(lower=0)
    penalty = -0.10 - 0.01 * shortfall          # -10%, and another -1% per missing ticker

    return weekly.fillna(penalty).mean()

In [143]:
result_high_returns = gp_minimize(

    lambda p: -score_high_return(p, tune_dates),
    hr_space, 
    n_calls=100,
    n_initial_points=10,
    acq_func="EI", 
    random_state=0, 
)

best_hr = result_high_returns.x 

# no cutoff comparison 
no_cut_hr = [-np.inf if d == ">" else np.inf for d in HIGH_RET_RULES.values()]


print("best cutoffs:", dict(zip(HIGH_RET_RULES, [round(float(x), 2) for x in best_hr])))
print()


print("                tune      val")
print("tuned:      ", round(score_high_return(best_hr, tune_dates), 4),
      "  ", round(score_high_return(best_hr, validation_dates), 4))
print("no cutoffs: ", round(score_high_return(no_cut_hr, tune_dates), 4),
      "  ", round(score_high_return(no_cut_hr, validation_dates), 4))
print()


tuned_weeks = weekly_high_return(best_hr, validation_dates)
base_weeks = weekly_high_return(no_cut_hr, validation_dates)

wins = (tuned_weeks > base_weeks).sum()
print(wins, "wins out of", len(validation_dates), "validation weeks")


print("tickers passing each validation week:",
      count_passing(high_return_tables, HIGH_RET_RULES, best_hr, validation_dates).tolist())

best cutoffs: {'adx 3M': 8.25, 'DI spread 3M': -8.83, 'adx 1M': 14.49, 'DI spread 1M': -17.87, 'Beta': 0.09, 'Market Cap': 3325336946082.17}

                tune      val
tuned:       -0.0038    0.0062
no cutoffs:  -0.0049    0.0066

15 wins out of 30 validation weeks
tickers passing each validation week: [20, 22, 24, 19, 17, 20, 16, 12, 16, 19, 16, 16, 14, 15, 18, 19, 20, 22, 20, 19, 20, 17, 15, 14, 13, 14, 15, 17, 18, 17]


In [144]:
def show_passing(cutoffs, dates, show_picks=True):
    for i in dates:
        passed = passes(high_return_tables, HIGH_RET_RULES, cutoffs, i)
        names = list(passed[passed].index)
        print(f"{i}  ({len(names)} passed): {names}")

        if show_picks and len(names) >= MIN_TICKERS:
            picks = high_return_tables["adx 3M"].loc[i, passed].nlargest(MAX_TICKERS).index
            dropped = [t for t in names if t not in picks]
            print(f"     held ({len(picks)}): {list(picks)}")
            if dropped:
                print(f"     passed but cut by top-{MAX_TICKERS}: {dropped}")
        print()
latest = high_return_tables["adx 3M"].dropna(how="all").index[-1]
show_passing(best_hr, [latest])

2026-11-23  (14 passed): ['ABBV', 'ACN', 'BAC', 'CL', 'KO', 'LLY', 'MO', 'MRK', 'PFE', 'RY.TO', 'PYPL', 'T.TO', 'UNP', 'UPS']
     held (14): ['T.TO', 'KO', 'ABBV', 'MRK', 'LLY', 'PFE', 'ACN', 'BAC', 'UNP', 'PYPL', 'CL', 'RY.TO', 'UPS', 'MO']



### PART D: LOW VOLATILITY 


In [152]:
# CONSTANTS
lv_window = 63
MAX_TICKERS = 20 #same variables just restating for clarity
MIN_TICKERS = 10 #same variables just restating for clarity
sector_cap = 0.4
max_per_sector = 8  

min_tickers = MIN_TICKERS  # made minimum 15 because portfolio volatility between 10 and 15 tickers are more or less the same. 
# Having it at 15 for sanity purposes lol .

#More testing determins that the volatility between 10 and 20 tickers by the strategy are more or less the same. 15 is in the sweet spot
# in the middle so yeah we go with 15. :)


sector_list = [
    "Utilities",
    "Consumer Defensive",
    "Healthcare",
    "Real Estate",
    "Financial Services",
    "Industrials",
    "Energy",
    "Basic Materials",
    "Communication Services",
    "Consumer Cyclical",
    "Technology"
]

In [153]:
def pull_stock_sectors (tickers):
    sectors = []

    for ticker in tickers:
        stock = yf.Ticker(ticker)
        sector = stock.info.get("sector")

        sectors.append({
            "Ticker": ticker,
            "Sector" :sector
        })
    return pd.DataFrame(sectors)


sectors = pull_stock_sectors(stock_data.keys())
print(sectors)

     Ticker                  Sector
0      AAPL              Technology
1      ABBV              Healthcare
2       ABT              Healthcare
3       ACN              Technology
4       AIG      Financial Services
5      AMZN       Consumer Cyclical
6       AXP      Financial Services
7        BA             Industrials
8       BAC      Financial Services
9     BB.TO              Technology
10     BIIB              Healthcare
11      BLK      Financial Services
12      BMY              Healthcare
13        C      Financial Services
14      CAT             Industrials
15       CL      Consumer Defensive
16       KO      Consumer Defensive
17      LLY              Healthcare
18      LMT             Industrials
19       MO      Consumer Defensive
20      MRK              Healthcare
21      PEP      Consumer Defensive
22      PFE              Healthcare
23    RY.TO      Financial Services
24       PM      Consumer Defensive
25     PYPL      Financial Services
26     QCOM              Tec

In [154]:
def lookback_returns(date, window=lv_window):
    window_returns = returns.loc[:date].tail(window)
    return window_returns

def portfolio_vol(window_returns, tickers):
    port_returns = window_returns[tickers].mean(axis=1)
    return port_returns.std()

sector_of = sectors.set_index("Ticker")["Sector"].to_dict()

def get_sector(ticker):
    s = sector_of.get(ticker)
    if s is None:
        return (f"Unknown_{ticker}")
    return s


def sector_counts(tickers):
    counts = {}
    for t in tickers:
        s = get_sector(t)
        counts[s] = counts.get(s, 0) + 1
    return counts

def over_cap_sectors(selected):
    if len(selected) == 0:
        return set()
    over = set()
    for s, c in sector_counts(selected).items():
        if c / len(selected) > sector_cap + 1e-9:
            over.add(s)
    return over

def usable_universe(window_returns, min_coverage=0.9):
    coverage = window_returns.notna().mean()
    return list(coverage[coverage >= min_coverage].index)

def eligible_candidates(selected, universe):
    counts = sector_counts(selected)
    over = over_cap_sectors(selected)

    base = []
    for t in universe:
        if t not in selected and counts.get(get_sector(t), 0) < max_per_sector:
            base.append(t)

    preferred = [t for t in base if get_sector(t) not in over]

    if len(preferred) > 0:
        return preferred
    return base 


In [155]:
def build_low_vol(date, universe=None, verbose=False, min_n= min_tickers, max_n=MAX_TICKERS):
    window = lookback_returns(date)

    if universe is None:
        universe = usable_universe(window)

    selected = []
    current_vol = np.inf 

    while len(selected) < MAX_TICKERS:
        candidates = eligible_candidates(selected, universe)
        if len(candidates) == 0:
            break

        best_ticker = None
        best_vol = np.inf
        for i in candidates:
            v = portfolio_vol(window, selected + [i])
            if pd.notna(v) and v < best_vol:
                best_vol = v
                best_ticker = i

        if best_ticker is None:
            break

        # STOPPING RULE: all three must be true to stop. subject to change depending on tests(don't forget leon)
        raises_vol = best_vol >= current_vol
        sectors_ok = len(selected) > 0 and len(over_cap_sectors(selected)) == 0
        big_enough = len(selected) >= min_tickers

        if raises_vol and sectors_ok and big_enough:
            if verbose:
                print(f"STOP: adding {best_ticker} would raise vol to {best_vol:.5f}")
            break

        selected.append(best_ticker)
        current_vol = best_vol

        if verbose:
            print(f"N={len(selected):2d}  added {best_ticker:8s} ({get_sector(best_ticker)})  vol={current_vol:.5f}")

    # warnings 
    if len(over_cap_sectors(selected)) > 0:
        print(f"WARNING {date}: sectors over cap: {over_cap_sectors(selected)}")
    if len(selected) < min_tickers:
        print(f"WARNING {date}: only {len(selected)} tickers selected (min {min_tickers})")

    return selected

In [156]:
def calculate_sector_weights(portfolio):
    return portfolio.groupby("Sector")["Weight"].sum()

def check_sector_limit(portfolio):
    sector_weights = calculate_sector_weights(portfolio)
    return (sector_weights <= sector_cap + 1e-9).all()


def make_low_vol_portfolio(selected):

    n = len(selected)
    if n == 0:
        print("ERROR: empty portfolio")
        return pd.DataFrame(columns=["Ticker", "Sector", "Weight"])

    portfolio = pd.DataFrame({
        "Ticker": selected,
        "Sector": [get_sector(t) for t in selected],
        "Weight": [1 / n] * n,
    })

    problems = []
    if not (min_tickers <= n <= MAX_TICKERS):
        problems.append(f"N={n} outside [{lv_min_n}, {lv_max_n}]")
    if abs(portfolio["Weight"].sum() - 1) > 1e-9:
        problems.append(f"weights sum to {portfolio['Weight'].sum()}")
    if portfolio["Weight"].min() < 0.05 - 1e-9 or portfolio["Weight"].max() > 0.15 + 1e-9:
        problems.append("a weight is outside 5%-15%")
    if not check_sector_limit(portfolio):
        problems.append(f"sector over 40%: {calculate_sector_weights(portfolio).to_dict()}")

    if problems:
        print("PORTFOLIO CHECK FAILED:")
        for p in problems:
            print("   -", p)
    else:
        print(f"All checks passed ({n} tickers, {1/n:.2%} each)")

    return portfolio

In [157]:
### For testing purposes, has no affect on actual portfolio result
'''
def realized_vol(tickers, date, hold=HOLD_DAYS):
    """Std of the equal-weight portfolio's daily returns over the NEXT `hold` days."""
    future = returns.loc[date:].iloc[1:hold + 1]
    return future[tickers].mean(axis=1).std()


std_table = build_tables({"Rolling STD 63d": "<"})["Rolling STD 63d"]

strategies = {
    "greedy (default)": lambda d: build_low_vol(d),
    "greedy forced 15": lambda d: build_low_vol(d, min_n=15, max_n=15),
    "greedy forced 20": lambda d: build_low_vol(d, min_n=20, max_n=20),
    "20 lowest std":    lambda d: list(std_table.loc[d].dropna().nsmallest(20).index),
}

# no tuning happens in low vol, so every test date is out-of-sample -> use all of them
bt_dates = test_dates

results = {name: [] for name in strategies}
sizes = {name: [] for name in strategies}

for d in bt_dates:
    for name, pick_fn in strategies.items():
        picks = pick_fn(d)
        results[name].append(realized_vol(picks, d))
        sizes[name].append(len(picks))

results = pd.DataFrame(results, index=bt_dates)
sizes = pd.DataFrame(sizes, index=bt_dates)

print("average realized 6-day vol (lower = better):")
print(results.mean().round(5))
print()
print("median realized vol:")
print(results.median().round(5))
print()
print("average portfolio size:")
print(sizes.mean().round(1))
print()

base = "greedy (default)"
for name in strategies:
    if name != base:
        wins = (results[base] < results[name]).sum()
        print(f"{base} beat {name}: {wins} / {len(bt_dates)} weeks")

'''

'\ndef realized_vol(tickers, date, hold=HOLD_DAYS):\n    """Std of the equal-weight portfolio\'s daily returns over the NEXT `hold` days."""\n    future = returns.loc[date:].iloc[1:hold + 1]\n    return future[tickers].mean(axis=1).std()\n\n\nstd_table = build_tables({"Rolling STD 63d": "<"})["Rolling STD 63d"]\n\nstrategies = {\n    "greedy (default)": lambda d: build_low_vol(d),\n    "greedy forced 15": lambda d: build_low_vol(d, min_n=15, max_n=15),\n    "greedy forced 20": lambda d: build_low_vol(d, min_n=20, max_n=20),\n    "20 lowest std":    lambda d: list(std_table.loc[d].dropna().nsmallest(20).index),\n}\n\n# no tuning happens in low vol, so every test date is out-of-sample -> use all of them\nbt_dates = test_dates\n\nresults = {name: [] for name in strategies}\nsizes = {name: [] for name in strategies}\n\nfor d in bt_dates:\n    for name, pick_fn in strategies.items():\n        picks = pick_fn(d)\n        results[name].append(realized_vol(picks, d))\n        sizes[name].appen

In [158]:
latest = returns.dropna(how="all").index[-1]
lv_picks = build_low_vol(latest, verbose=True)
lv_portfolio = make_low_vol_portfolio(lv_picks)
print(lv_portfolio)
print(calculate_sector_weights(lv_portfolio))

N= 1  added KO       (Consumer Defensive)  vol=0.01162
N= 2  added RY.TO    (Financial Services)  vol=0.00881
N= 3  added PFE      (Healthcare)  vol=0.00861
N= 4  added AAPL     (Technology)  vol=0.00867
N= 5  added LMT      (Industrials)  vol=0.00861
N= 6  added AMZN     (Consumer Cyclical)  vol=0.00823
N= 7  added T.TO     (Communication Services)  vol=0.00726
N= 8  added ABBV     (Healthcare)  vol=0.00713
N= 9  added MO       (Consumer Defensive)  vol=0.00715
N=10  added CAT      (Industrials)  vol=0.00691
STOP: adding PEP would raise vol to 0.00695
All checks passed (10 tickers, 10.00% each)
  Ticker                  Sector  Weight
0     KO      Consumer Defensive     0.1
1  RY.TO      Financial Services     0.1
2    PFE              Healthcare     0.1
3   AAPL              Technology     0.1
4    LMT             Industrials     0.1
5   AMZN       Consumer Cyclical     0.1
6   T.TO  Communication Services     0.1
7   ABBV              Healthcare     0.1
8     MO      Consumer Defen

my portfolio volatilty says that the lowest volatility in the past 63 days has 10 tickers, going above 10 tickers increases the volatility number. Should I just trust the code and strategy and just have it so that I don't force it to have higher than the min number of tickers, i can set it to force have 15 tickers. Or do I force it to have 15 tickers, knowing that in testing the volatility in the past 63 days is lowest at 10 tickers. 